In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import *

# Create Spark session
spark = SparkSession.builder \
    .appName("Day3_Window_Functions") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print(f"✅ Spark session created: {spark.version}")

26/03/10 17:19:51 WARN Utils: Your hostname, odinsbeard-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/03/10 17:19:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 17:19:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/10 17:20:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/10 17:20:09 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


✅ Spark session created: 3.5.0


In [2]:
# Load data
df_sales = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv("file:///home/odinsbeard/Data_engineering_Journey/week6_spark/data/input/sales_*.csv")

df_products = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv("file:///home/odinsbeard/Data_engineering_Journey/week6_spark/data/input/products_*.csv")

df_users = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv("file:///home/odinsbeard/Data_engineering_Journey/week6_spark/data/input/users_*.csv")


# Register views
df_sales.createOrReplaceTempView("sales")
df_products.createOrReplaceTempView("products")
df_users.createOrReplaceTempView("users")


print(f"✅ Data loaded: {df_sales.count():,} sales rows")
print(f"✅ Data loaded: {df_products.count():,} products rows")
print(f"✅ Data loaded: {df_users.count():,} users rows")

✅ Data loaded: 20,000,000 sales rows


✅ Data loaded: 400,000 products rows


[Stage 12:=============================>                            (3 + 3) / 6]

✅ Data loaded: 4,000,000 users rows


In [3]:
# Running total of revenue by category over time
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, col

# First, get daily revenue by category
daily_category_revenue = df_sales.join(df_products, "product_id") \
    .groupBy("sale_date", "category") \
    .agg(sum("final_amount").alias("daily_revenue")) \
    .orderBy("sale_date")

# Define window for running total within each category
window_spec = Window.partitionBy("category") \
    .orderBy("sale_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Add running total
df_with_running = daily_category_revenue \
    .withColumn("running_total", sum("daily_revenue").over(window_spec))

print("📊 Running total by category:")
df_with_running.show(20)



📊 Running total by category:


[Stage 27:>                                                         (0 + 1) / 1]

+----------+--------+------------------+--------------------+
| sale_date|category|     daily_revenue|       running_total|
+----------+--------+------------------+--------------------+
|2023-01-01|   Books|1431823.3700000003|  1431823.3700000003|
|2023-01-02|   Books|        1481323.78|  2913147.1500000004|
|2023-01-03|   Books|1469911.9899999993|          4383059.14|
|2023-01-04|   Books|1535954.1399999997|   5919013.279999999|
|2023-01-05|   Books|1487345.0000000005|   7406358.279999999|
|2023-01-06|   Books|1467624.2800000003|   8873982.559999999|
|2023-01-07|   Books|1528544.8800000001|       1.040252744E7|
|2023-01-08|   Books|        1481338.42|       1.188386586E7|
|2023-01-09|   Books|         1495055.4|       1.337892126E7|
|2023-01-10|   Books|1494224.5300000003|       1.487314579E7|
|2023-01-11|   Books|        1458587.46|       1.633173325E7|
|2023-01-12|   Books|1475700.4999999995|       1.780743375E7|
|2023-01-13|   Books|        1519170.69|       1.932660444E7|
|2023-01

In [4]:
# Find top 3 products by revenue in each category
from pyspark.sql.functions import rank, desc

# Calculate product revenue
product_revenue = df_sales.groupBy("product_id") \
    .agg(sum("final_amount").alias("revenue"))

# Join with products
product_performance = product_revenue.join(df_products, "product_id") \
    .select("category", "product_name", "revenue")

# Define window for ranking within category
window_spec = Window.partitionBy("category") \
    .orderBy(desc("revenue"))

# Add rank
ranked_products = product_performance \
    .withColumn("rank", rank().over(window_spec))

# Filter top 3
top_products = ranked_products.filter(col("rank") <= 3) \
    .orderBy("category", "rank")

print("🏆 Top 3 products per category:")
top_products.show(30, truncate=False)



🏆 Top 3 products per category:


[Stage 32:======================================>                   (4 + 2) / 6]

+-----------+---------------+-----------------+----+
|category   |product_name   |revenue          |rank|
+-----------+---------------+-----------------+----+
|Books      |Magazine Plus  |77767.96999999999|1   |
|Books      |Cookbook Lite  |74422.99         |2   |
|Books      |Fiction Max    |74091.45000000001|3   |
|Clothing   |Dress Plus     |74523.43000000001|1   |
|Clothing   |Socks Basic    |74308.88         |2   |
|Clothing   |Shorts Pro     |74246.84000000001|3   |
|Electronics|Headphones Max |75724.17         |1   |
|Electronics|Mouse Plus     |74313.68         |2   |
|Electronics|Smartphone Plus|73055.75         |3   |
|Home       |Tools Pro      |75769.23999999999|1   |
|Home       |Tools Max      |75025.66999999998|2   |
|Home       |Chair Plus     |74606.01999999999|3   |
|Sports     |Dumbbells Pro  |74631.66999999998|1   |
|Sports     |Jump Rope Basic|72543.52         |2   |
|Sports     |Football Basic |72289.1          |3   |
+-----------+---------------+-----------------

In [6]:
# Find each customer's first and last purchase
from pyspark.sql.functions import first, last, row_number

# Define window for customer purchases
window_spec = Window.partitionBy("user_id") \
    .orderBy("sale_date") \
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

# Get first and last purchase
customer_first_last = df_sales.select("user_id", "sale_date", "final_amount") \
    .withColumn("first_purchase_date", first("sale_date").over(window_spec)) \
    .withColumn("last_purchase_date", last("sale_date").over(window_spec)) \
    .withColumn("first_amount", first("final_amount").over(window_spec)) \
    .withColumn("last_amount", last("final_amount").over(window_spec)) \
    .withColumn("purchase_number", row_number().over(
        Window.partitionBy("user_id").orderBy("sale_date")))

# Get only first row for each customer
customer_summary = customer_first_last.filter(col("purchase_number") == 1) \
    .select("user_id", "first_purchase_date", "last_purchase_date", 
            "first_amount", "last_amount") \
    .join(df_users.select("user_id", "first_name", "last_name", "country"), "user_id")

print("👤 Customer first and last purchase:")
customer_summary.show(20, truncate=False)



👤 Customer first and last purchase:


[Stage 46:>                                                         (0 + 1) / 1]

+-------+-------------------+------------------+------------+-----------+----------+---------+---------+
|user_id|first_purchase_date|last_purchase_date|first_amount|last_amount|first_name|last_name|country  |
+-------+-------------------+------------------+------------+-----------+----------+---------+---------+
|12     |2023-01-23         |2023-01-23        |374.85      |374.85     |Michelle  |Taylor   |USA      |
|22     |2023-03-22         |2023-03-22        |691.26      |691.26     |Anthony   |Brown    |USA      |
|26     |2023-02-03         |2023-02-03        |16.36       |16.36      |Kenneth   |Moore    |Brazil   |
|27     |2023-03-08         |2023-03-08        |266.83      |266.83     |Barbara   |Miller   |UK       |
|28     |2023-08-16         |2023-08-16        |458.4       |458.4      |Donald    |Gonzalez |Canada   |
|31     |2023-03-25         |2023-03-25        |178.92      |178.92     |Charles   |Ramirez  |UK       |
|34     |2023-03-18         |2023-03-18        |211.26 

In [7]:
# Calculate month-over-month growth percentage
from pyspark.sql.functions import month, year, lag, round

# Extract year and month
df_with_month = df_sales.withColumn("year", year("sale_date")) \
    .withColumn("month", month("sale_date"))

# Monthly revenue
monthly_revenue = df_with_month.groupBy("year", "month") \
    .agg(sum("final_amount").alias("monthly_revenue")) \
    .orderBy("year", "month")

# Define window to get previous month
window_spec = Window.orderBy("year", "month")

# Calculate growth
monthly_growth = monthly_revenue \
    .withColumn("prev_month_revenue", lag("monthly_revenue", 1).over(window_spec)) \
    .withColumn("growth_amount", 
                col("monthly_revenue") - col("prev_month_revenue")) \
    .withColumn("growth_percent", 
                round((col("monthly_revenue") - col("prev_month_revenue")) / 
                      col("prev_month_revenue") * 100, 2))

print("📈 Month-over-month growth:")
monthly_growth.show(24)



📈 Month-over-month growth:


26/03/10 17:48:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/10 17:48:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/10 17:48:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/10 17:49:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/10 17:49:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/10 17:49:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/10 1

+----+-----+--------------------+--------------------+--------------------+--------------+
|year|month|     monthly_revenue|  prev_month_revenue|       growth_amount|growth_percent|
+----+-----+--------------------+--------------------+--------------------+--------------+
|2023|    1|2.5978428894999814E8|                NULL|                NULL|          NULL|
|2023|    2|2.3479099574999988E8|2.5978428894999814E8|-2.499329319999826E7|         -9.62|
|2023|    3|2.6052160238999927E8|2.3479099574999988E8| 2.573060663999939E7|         10.96|
|2023|    4|2.5201717100999963E8|2.6052160238999927E8|  -8504431.379999638|         -3.26|
|2023|    5|2.5950260255999896E8|2.5201717100999963E8|  7485431.5499993265|          2.97|
|2023|    6|2.5198493718999997E8|2.5950260255999896E8| -7517665.3699989915|          -2.9|
|2023|    7|2.5960201244999993E8|2.5198493718999997E8|   7617075.259999961|          3.02|
|2023|    8|2.5975428281999946E8|2.5960201244999993E8|  152270.36999952793|          0.06|

In [8]:

# Segment customers into spending tiers
from pyspark.sql.functions import ntile, when

# Calculate total spent per customer
customer_spend = df_sales.groupBy("user_id") \
    .agg(sum("final_amount").alias("total_spent"))

# Define window for ntile
window_spec = Window.orderBy(col("total_spent").desc())

# Create segments
customer_segments = customer_spend \
    .withColumn("tier", ntile(5).over(window_spec)) \
    .withColumn("segment", 
                when(col("tier") == 1, "VIP") \
                .when(col("tier") == 2, "Gold") \
                .when(col("tier") == 3, "Silver") \
                .when(col("tier") == 4, "Bronze") \
                .otherwise("Basic"))

# Add user details
customer_analysis = customer_segments.join(
    df_users.select("user_id", "first_name", "last_name", "country"), "user_id") \
    .select("user_id", "first_name", "last_name", "country", 
            "total_spent", "segment", "tier") \
    .orderBy(desc("total_spent"))

print("💎 Customer segmentation:")
customer_analysis.show(20, truncate=False)

# Summary by segment
segment_summary = customer_analysis.groupBy("segment") \
    .agg(count("*").alias("customer_count"),
         avg("total_spent").alias("avg_spent"),
         sum("total_spent").alias("total_revenue")) \
    .orderBy(desc("avg_spent"))

print("\n📊 Segment summary:")
segment_summary.show()



💎 Customer segmentation:


26/03/10 18:03:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/10 18:03:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/10 18:03:41 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/10 18:03:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/10 18:03:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/10 18:03:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/10 18:03:48 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/10 18:03:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatc

+-------+----------+---------+---------+------------------+-------+----+
|user_id|first_name|last_name|country  |total_spent       |segment|tier|
+-------+----------+---------+---------+------------------+-------+----+
|749090 |Paul      |White    |Canada   |17409.94          |VIP    |1   |
|462724 |Linda     |Perez    |Germany  |17142.46          |VIP    |1   |
|920754 |Donna     |Wright   |UK       |17081.070000000003|VIP    |1   |
|222556 |Lisa      |Flores   |Australia|16854.16          |VIP    |1   |
|327485 |Paul      |Sanchez  |Australia|16772.420000000002|VIP    |1   |
|201737 |Steven    |Nguyen   |Japan    |16751.579999999998|VIP    |1   |
|636879 |Sarah     |Wright   |Brazil   |16748.269999999997|VIP    |1   |
|16421  |Michelle  |Martin   |UK       |16748.05          |VIP    |1   |
|557207 |Donald    |Gonzalez |France   |16684.13          |VIP    |1   |
|668083 |Richard   |Williams |Germany  |16533.13          |VIP    |1   |
|686552 |Barbara   |Gonzalez |USA      |16447.42000

26/03/10 18:06:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/10 18:06:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/10 18:06:48 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/10 18:06:48 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/10 18:06:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/10 18:06:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/10 18:06:52 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/10 18:06:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatc

+-------+--------------+------------------+--------------------+
|segment|customer_count|         avg_spent|       total_revenue|
+-------+--------------+------------------+--------------------+
|    VIP|        730969| 5857.337654551593|4.2815322480099235E9|
|   Gold|        730968| 3281.589540964823|2.3987369435799747E9|
| Silver|        730968| 1983.379586178886| 1.449787009350008E9|
| Bronze|        730968|1076.1394458170853| 7.866234984300232E8|
|  Basic|        730968| 364.2761366297627|2.6627419903998435E8|
+-------+--------------+------------------+--------------------+



In [10]:
# Find products often bought together in same sale/order
from pyspark.sql.functions import collect_list, size

# Group products by sale_id (each sale/transaction)
order_products = df_sales.groupBy("sale_id") \
    .agg(collect_list("product_id").alias("products")) \
    .filter(size("products") > 1)  # Sales with multiple items

# Show what we have
print(f"Found {order_products.count():,} sales with multiple items")
order_products.show(5, truncate=False)

# Create a temporary view for SQL
order_products.createOrReplaceTempView("order_products")

# Use SQL to find product pairs
product_pairs = spark.sql("""
    SELECT 
        a.product_id as product1,
        b.product_id as product2,
        COUNT(*) as times_together
    FROM order_products o
    LATERAL VIEW explode(o.products) a AS product_id
    LATERAL VIEW explode(o.products) b AS product_id
    WHERE a.product_id < b.product_id
    GROUP BY a.product_id, b.product_id
    ORDER BY times_together DESC
    LIMIT 20
""")

# Add product names
product_pairs_with_names = product_pairs \
    .join(df_products.selectExpr("product_id as product1", "product_name as product1_name"), "product1") \
    .join(df_products.selectExpr("product_id as product2", "product_name as product2_name"), "product2") \
    .select("product1_name", "product2_name", "times_together")

print("\n🛒 Products frequently bought together:")
product_pairs_with_names.show(truncate=False)



Found 0 sales with multiple items


[Stage 113:============================>                            (1 + 1) / 2]

+-------+--------+
|sale_id|products|
+-------+--------+
+-------+--------+




🛒 Products frequently bought together:


[Stage 117:========================================>                (5 + 2) / 7]

+-------------+-------------+--------------+
|product1_name|product2_name|times_together|
+-------------+-------------+--------------+
+-------------+-------------+--------------+

